# TP 6 — Mini-projet : MLP sur données tabulaires

**Objectif.** Aucune notion nouvelle en cours aujourd'hui : ce TP mobilise tout ce que vous avez appris depuis la séance 3 (MLP, `Trainer`, choix de loss, régularisation) sur un jeu de données tabulaire réaliste. Nouveauté du jour : l'encodage *one-hot* des variables catégorielles et la standardisation des variables numériques sont cette fois à votre charge.

**Dataset.** Titanic (`sklearn.datasets.fetch_openml`) : on prédit la survie d'un passager (classification binaire) à partir de variables numériques (âge, tarif, nombre de proches à bord) et catégorielles (classe du billet, sexe, port d'embarquement).

Le chargement et le nettoyage des données (valeurs manquantes) sont fournis ci-dessous ; à vous d'encoder et de standardiser les variables (partie 1), de définir et d'entraîner un premier modèle (partie 2), d'en tester l'amélioration par batch normalization et régularisation (partie 3), puis de rechercher de bons hyperparamètres (partie 4).

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import itertools

from training_toolbox import Trainer, EarlyStopping, ModelCheckpoint

torch.manual_seed(0)

## Chargement et nettoyage des données (fourni)

In [ ]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = titanic.frame

numeric_cols = ["age", "fare", "sibsp", "parch"]
categorical_cols = ["pclass", "sex", "embarked"]
target_col = "survived"

df = df[numeric_cols + categorical_cols + [target_col]].copy()
df[target_col] = df[target_col].astype(str).astype(int)

# Imputation très simple : médiane pour le numérique, modalité la plus fréquente pour le catégoriel
for col in numeric_cols:
    df[col] = df[col].astype(float).fillna(df[col].astype(float).median())
for col in categorical_cols:
    df[col] = df[col].astype("object")
    df[col] = df[col].fillna(df[col].mode()[0])

print("Valeurs manquantes par colonne (après nettoyage) :")
print(df.isna().sum())
print(df.head())

## Partie 1 — Encodage et standardisation

**Question 1.1.** Encodez les variables catégorielles en *one-hot* (une colonne indicatrice 0/1 par modalité), puis construisez la matrice de features complète `X` (`numpy.ndarray` de type `float32`) en concaténant les colonnes numériques et les colonnes one-hot. Récupérez également le vecteur cible `y` (`float32`).

In [ ]:
# TODO : df_onehot = pd.get_dummies(df[categorical_cols], columns=categorical_cols)
#        X = concaténation de df[numeric_cols] et df_onehot, convertie en numpy float32
#        y = df[target_col], converti en numpy float32

**Question 1.2.** Séparez `X`/`y` en train/val, standardisez `X`, puis construisez un `TensorDataset`/`DataLoader` classique comme dans les séances précédentes.

In [ ]:
# TODO : train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
#        StandardScaler ajusté sur X_train, appliqué à X_train et X_val
#        train_loader = DataLoader(TensorDataset(...), batch_size=32, shuffle=True)
#        val_loader = DataLoader(TensorDataset(...), batch_size=64, shuffle=False)
#        n_features = X_train.shape[1]

In [ ]:
def binary_accuracy(preds, y):
    pred_labels = (preds > 0).float()
    return (pred_labels == y.view(-1, 1)).float().mean()

## Partie 2 — Un premier MLP

**Question 2.1.** Définissez `TabularMLP(n_features, hidden_sizes=(64, 32))` : un MLP (`nn.Sequential`) qui empile, pour chaque taille de `hidden_sizes`, une couche `nn.Linear` suivie d'une activation `nn.ReLU`, puis une dernière `nn.Linear` de sortie.

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, n_features, hidden_sizes=(64, 32)):
        super().__init__()
        # TODO : empiler nn.Linear -> nn.ReLU pour chaque taille de hidden_sizes,
        #        puis une dernière nn.Linear vers 1 sortie (sans activation)

    def forward(self, x):
        # TODO
        pass

**Question 2.2.** Écrivez une fonction `build_and_train(hidden_sizes=(64, 32), lr=1e-3, epochs=30, verbose=False, callbacks=None)` qui instancie un `TabularMLP`, l'entraîne avec le `Trainer` sur `train_loader`/`val_loader`, et renvoie `(model, history)`. Appelez-la avec les valeurs par défaut (`verbose=True`), tracez les courbes de perte et d'accuracy (train/val), et affichez l'accuracy de validation finale.

In [ ]:
def build_and_train(hidden_sizes=(64, 32), lr=1e-3, epochs=30, verbose=False, callbacks=None):
    # TODO : instancier TabularMLP(n_features, hidden_sizes), un optimizer Adam(lr=lr),
    #        un Trainer (nn.BCEWithLogitsLoss(), metrics={"acc": binary_accuracy}, callbacks=callbacks)
    #        puis appeler trainer.fit(train_loader, val_loader, epochs=epochs, verbose=verbose)
    #        et renvoyer (model, history)
    pass


# TODO : model, history_baseline = build_and_train(verbose=True)
#        tracer history_baseline["train_loss"]/["val_loss"] puis ["train_acc"]/["val_acc"]
#        et afficher history_baseline["val_acc"][-1]

## Partie 3 — Batch normalization et régularisation

**Question 3.1.** Définissez `TabularMLPBatchNorm`, identique à `TabularMLP` mais avec une couche `nn.BatchNorm1d(hidden_size)` insérée entre chaque couche linéaire cachée et son activation `ReLU`. Entraînez-le avec les mêmes hyperparamètres que la partie 2 et comparez sa courbe de `val_loss` à celle de `history_baseline`.

In [ ]:
class TabularMLPBatchNorm(nn.Module):
    def __init__(self, n_features, hidden_sizes=(64, 32)):
        super().__init__()
        # TODO : mêmes couches que TabularMLP, avec un nn.BatchNorm1d(hidden_size) entre
        #        chaque couche linéaire cachée et sa ReLU

    def forward(self, x):
        # TODO
        pass


# TODO : entraîner TabularMLPBatchNorm avec les mêmes hyperparamètres que la partie 2 -> history_bn
#        puis tracer history_baseline["val_loss"] et history_bn["val_loss"] sur un même graphique

**Question 3.2.** Comparez les courbes train/val de `history_baseline` (partie 2) : le modèle sur-apprend-il ? Si oui, choisissez, parmi les leviers vus en séance 5, celui ou ceux qui vous semblent les plus adaptés ici, mettez-le(s) en œuvre (en adaptant votre architecture et/ou votre fonction d'entraînement selon la technique choisie), et comparez vos résultats à `history_baseline`.

In [ ]:
# TODO : le modèle de la partie 2 sur-apprend-il (comparer train_loss/val_loss de history_baseline) ?
#        Si oui, choisissez un ou plusieurs leviers de régularisation vus en séance 5, implémentez-les
#        (nouvelle classe de modèle et/ou nouveaux arguments de l'optimizer/du Trainer selon le levier
#        choisi), entraînez -> history_reg, et comparez ses courbes à celles de history_baseline

## Partie 4 — Recherche d'hyperparamètres

**Question 4.1.** Complétez la boucle ci-dessous pour évaluer chaque combinaison de `param_grid` (`hidden_sizes`, `lr`) avec `build_and_train` (nombre d'epochs réduit pour que la recherche reste rapide, par exemple `epochs=15`), et stockez l'accuracy de validation finale de chaque combinaison dans `results`.

In [ ]:
param_grid = {
    "hidden_sizes": [(32,), (64, 32), (128, 64)],
    "lr": [1e-3, 1e-2],
}

results = []
for hidden_sizes, lr in itertools.product(param_grid["hidden_sizes"], param_grid["lr"]):
    # TODO : model, history = build_and_train(hidden_sizes=hidden_sizes, lr=lr, epochs=15)
    #        val_acc = history["val_acc"][-1]
    #        results.append((hidden_sizes, lr, val_acc))
    pass

results.sort(key=lambda r: r[-1], reverse=True)
for r in results:
    print(r)

**Question 4.2.** Reprenez la meilleure combinaison trouvée, et réentraînez-la seule avec davantage d'epochs (par exemple 60) et les callbacks `EarlyStopping`/`ModelCheckpoint` vus en séance 5, pour obtenir un modèle final.

In [ ]:
# TODO : reprendre les meilleurs hyperparamètres (results[0]), puis appeler build_and_train
#        avec epochs=60 et callbacks=[EarlyStopping(patience=5), ModelCheckpoint("best_titanic.pt")]

**Questions.**
- Le modèle final (partie 4) est-il vraiment meilleur que celui de la partie 2, ou l'essentiel du gain vient-il de la régularisation testée en partie 3 ?
- La recherche d'hyperparamètres a-t-elle changé le classement des modèles par rapport à vos choix « par défaut » ? Qu'auriez-vous pu explorer d'autre (dropout, weight decay, nombre de couches, taille de batch) ?

_Votre réponse ici._

## Bilan

Ce mini-projet clôt le bloc « MLP » du cours (séances 3 à 6) : architecture, optimisation, choix de loss, prétraitement de données tabulaires (imputation, one-hot encoding), régularisation et recherche d'hyperparamètres. Les séances 7 à 9 passent aux CNN, pour des données image.